# Imports

In [1]:
import numpy as np
from sklearn.svm import SVC, SVR
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris, load_diabetes
from sklearn.metrics import mean_squared_error, accuracy_score

random_state = 42
np.random.seed(random_state)

# Kernel Definition

In [2]:
# Define a simple single-hidden-layer neural network
def relu(x):
    return np.maximum(0, x)

def forward_pass(W1, W2, x):
    """Compute the network output."""
    hidden = relu(np.dot(W1, x))  # First layer activation
    output = np.dot(W2, hidden)   # Output layer
    return hidden, output

In [3]:
def compute_ntk(X1, W1, W2, X2=None):
    if X2 is None:
        X2 = X1  # If X2 is not provided, compute NTK for X1 only

    N1 = X1.shape[1]  # Number of samples in X1
    N2 = X2.shape[1]  # Number of samples in X2
    m, d = W1.shape   # Number of hidden neurons and input dimension

    # Initialize NTK matrix
    NTK = np.zeros((N1, N2))

    # Compute gradients for each pair of input samples
    for i in range(N1):
        for j in range(N2):
            x_i = X1[:, i]
            x_j = X2[:, j]

            # Forward pass
            h_i, f_i = forward_pass(W1, W2, x_i)
            h_j, f_j = forward_pass(W1, W2, x_j)

            # Compute gradients
            dh_dW1_i = np.outer((h_i > 0).astype(float), x_i)  # ReLU derivative
            dh_dW1_j = np.outer((h_j > 0).astype(float), x_j)

            df_dW1_i = np.outer(W2, dh_dW1_i)  # Gradient w.r.t. W1
            df_dW1_j = np.outer(W2, dh_dW1_j)

            df_dW2_i = h_i.reshape(-1, 1)  # Gradient w.r.t. W2
            df_dW2_j = h_j.reshape(-1, 1)

            # Compute NTK entry (sum of gradient inner products)
            NTK[i, j] = np.sum(df_dW1_i * df_dW1_j) + np.sum(df_dW2_i * df_dW2_j)

    return NTK

# Classification (Iris):

In [4]:
iris = load_iris()
X_clf, y_clf = iris.data, iris.target
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=random_state)

In [5]:
W1_clf = np.random.randn(10, X_clf.shape[1])  # 10 hidden neurons
W2_clf = np.random.randn(10)

NTK_clf_train = compute_ntk(X_clf_train.T, W1_clf, W2_clf)
NTK_clf_test = compute_ntk(X_clf_test.T, W1_clf, W2_clf, X_clf_train.T)

svc_clf = SVC(kernel="precomputed")
svc_clf.fit(NTK_clf_train, y_clf_train)

y_clf_pred = svc_clf.predict(NTK_clf_test)
print(f"Mean Squared Error (Iris): {mean_squared_error(y_clf_test, y_clf_pred)}")
print("Classification Accuracy (Iris):", accuracy_score(y_clf_test, y_clf_pred))

Mean Squared Error (Iris): 0.0
Classification Accuracy (Iris): 1.0


# Regression (Diabetes):

In [6]:
diabetes = load_diabetes()
X_reg, y_reg = diabetes.data, diabetes.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [7]:
W1_reg = np.random.randn(50, X_reg.shape[1])  # 100 hidden neurons
W2_reg = np.random.randn(50)

NTK_reg_train = compute_ntk(X_reg_train.T, W1_reg, W2_reg)
NTK_reg_test = compute_ntk(X_reg_test.T, W1_reg, W2_reg, X_reg_train.T)

svr_reg = SVR(kernel="precomputed", C=1.0, epsilon=0.1)
svr_reg.fit(NTK_reg_train, y_reg_train)

y_reg_pred = svr_reg.predict(NTK_reg_test)
print("SVR Regression MSE (Diabetes):", mean_squared_error(y_reg_test, y_reg_pred))
print("SVR Regression R^2 (Diabetes):", svr_reg.score(NTK_reg_test, y_reg_test))

SVR Regression MSE (Diabetes): 2724.5717609337944
SVR Regression R^2 (Diabetes): 0.4857505239073925
